In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from utils.modelling import *
from utils.plotting import *

In [ ]:
import os
if os.path.exists("../data/claims_evaluation.parquet"):
    evaluation_df = pd.read_parquet("../data/claims_evaluation.parquet")
else:
    evaluation_df = pd.DataFrame()

## VERSION 1

In [ ]:
insurance_claims = pd.read_parquet("../data/claims_model_dataset_v1.parquet")

numeric_columns = insurance_claims.select_dtypes(include = ["int64", "int32", "float64"]).columns.drop("total_claim_amount")
categorical_columns = insurance_claims.select_dtypes(include = ["object"]).columns

insurance_claims = encode_features(insurance_claims, categorical_columns)
x_train, x_test, y_train, y_test = split_dataset(insurance_claims, "total_claim_amount", 0.2)

preprocessor = ColumnTransformer(
    transformers = [
        ("numeric_scaled", StandardScaler(), numeric_columns)
    ], 
    remainder = "passthrough"
)

#### LINEAR REGRESSION

In [ ]:
pipeline = create_pipeline(LinearRegression(), preprocessor)
lr = transform_target(pipeline, np.log1p, np.expm1)
lr.fit(x_train, y_train)
lr_results = regression_results(lr, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Linear Regression", "Initial", lr_results)

#### RANDOM FOREST

In [ ]:
pipeline = create_pipeline(RandomForestRegressor(random_state = 123), preprocessor)
rfr = transform_target(pipeline, np.log1p, np.expm1)
rfr.fit(x_train, y_train)
rfr_results = regression_results(rfr, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Random Forest", "Initial", rfr_results)

In [ ]:
parameters = {
    "regressor__model__n_estimators": [300], 
    "regressor__model__min_samples_split": [3],
    "regressor__model__min_samples_leaf": [4], 
    "regressor__model__max_depth": [3],
    "regressor__model__max_features": [0.75]
}

rfr_gs = optimise_model(rfr, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
rfr_gs_results = regression_results(rfr_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Random Forest", "Optimised", rfr_gs_results)

#### GRADIENT BOOSTING

In [ ]:
pipeline = create_pipeline(GradientBoostingRegressor(random_state = 123), preprocessor)
gbr = transform_target(pipeline, np.log1p, np.expm1)
gbr.fit(x_train, y_train)
gbr_results = regression_results(gbr, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Gradient Boosting", "Initial", gbr_results)

In [ ]:
parameters = {
    'regressor__model__learning_rate': [0.01],
    'regressor__model__n_estimators': [350],
    'regressor__model__max_depth': [3],
    'regressor__model__min_samples_split': [3],
    'regressor__model__min_samples_leaf': [3]
}

gbr_gs = optimise_model(gbr, parameters, "neg_root_mean_squared_error", 5, x_train, y_train)
gbr_gs_results = regression_results(gbr_gs, x_train, x_test, y_train, y_test)

#evaluation_df = store_regression_results(evaluation_df, "../data/claims_evaluation.parquet", "V1", "Gradient Boosting", "Optimised", gbr_gs_results)

### COMPARISON

In [ ]:
v1_comparison = evaluation_df[evaluation_df["dataset_version"] == "V1"]
v1_comparison.loc[[0, 2, 4]]

In [ ]:
dict_for_df = {
    "Linear Regression": lr.regressor_.named_steps["model"].coef_[0],
    "Random Forest": rfr_gs.regressor_.named_steps["model"].feature_importances_,
    "Gradient Boosting": gbr_gs.regressor_.named_steps["model"].feature_importances_
}

plot_feature_importance(x_train, dict_for_df, 0, 10)

## VERSION 2